# 12 — Observability with Tempo / OpenTelemetry

Demonstrates distributed tracing for the agent framework via **OpenTelemetry + Grafana Tempo**.

Every agent run, tool call, and LLM request is captured as a span and exported  
to a Tempo backend over OTLP (HTTP).

**Stack**:
- `opentelemetry-sdk` — spans and exporters
- `opentelemetry-exporter-otlp-proto-http` — OTLP/HTTP to Grafana Tempo
- Grafana at `http://localhost:3001` (started via `docker compose -f docker/docker-compose.yml`)

**Prerequisites**:
- `docker compose -f docker/docker-compose.yml up -d tempo grafana` (or your OTLP-compatible backend)
- `OPENAI_API_KEY` set

In [1]:
import os
import asyncio

OTLP_ENDPOINT = os.environ.get('OTEL_EXPORTER_OTLP_TRACES_ENDPOINT', 'localhost:4318')
print(f'OTLP endpoint: {OTLP_ENDPOINT}')

OTLP endpoint: localhost:4318


In [2]:
from agent_framework.core.agents.react_agent import ReActAgent
from agent_framework.core.tools.builtin_tools import CalculatorTool, GetCurrentTimeTool
from agent_framework.providers.llm.openai.openai_client import OpenAIClient
from agent_framework.core.memory.unbounded_memory import UnboundedMemory
from agent_framework.core.context.implementations import UnboundedContext
from agent_framework.core.console import Console
from agent_framework.runtime.observability.telemetry import configure_opentelemetry

## Configure OpenTelemetry

In [3]:
configure_opentelemetry(
    service_name='agent-framework-tempo-demo',
    otlp_trace_endpoint=OTLP_ENDPOINT,
)
print('OpenTelemetry configured ✅')

Configuring OpenTelemetry
Using OTLP HTTP trace exporter → http://localhost:4318/v1/traces
Metrics export is disabled (no OTLP metric endpoint and console export disabled)
OpenTelemetry configured ✅


## Run a traced agent

After this cell completes, open **Grafana → Explore → Tempo** and filter  
by service name `agent-framework-tempo-demo` to see the trace.

In [4]:
async def run():
    agent = ReActAgent(
        name='TempoDemoBot',
        description='A helpful assistant for demonstrating tracing.',
        model_client=OpenAIClient(model='gpt-4o'),
        tools=[CalculatorTool(), GetCurrentTimeTool()],
        memory=UnboundedMemory(),
        model_context=UnboundedContext(),
        max_iterations=5,
        verbose=True,
    )

    result = await Console(agent).run('Calculate 123 * 456 and tell me the current time.')

    # Give the OTLP exporter time to flush
    await asyncio.sleep(2)
    print('\nTraces flushed — check Grafana at http://localhost:3001')

await run()

You → Calculate 123 * 456 and tell me the current time.

╭───────────────────────────────────────────────── TempoDemoBot ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  The result of (123 \times 456) is (56,088).                                                                    │
│                                                                                                                 │
│  The current time in UTC is 19:12 on March 15, 2026.                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

completed · 2 steps · 2 tool calls · 511 tokens · 3.2s


Traces flushed — check Grafana at http://localhost:3001
